# Autoship Nudge Promo Incentive — Power Analysis (Autoship Adoption Rate, 2-Cell Design, Nudge-Reach Population)

**Experiment:** Autoship Nudge Promo Incentive Test · **Owner:** Sergio Oyola · **Primary metric analyzed here:** Autoship Adoption Rate · **Randomization unit:** `client_id` · **Allocation point:** post-First-Fix checkout, once keep rate is known, at the moment the client selects a Quick Fix date and clicks "Schedule a Quick Fix" (prior to Narvar handoff)

This notebook sizes a standard 2-cell A/B version of the experiment for its primary metric, **Autoship Adoption Rate**, using a population definition refined to reflect who actually reaches the allocation moment — not just who is broadly eligible for the test.

## Design: 2-cell test, single comparison
Eligible clients are randomized into 2 cells at a 50/50 split:

| Cell | Experience | Offer |
|---|---|---|
| Control | BAU Quick Fix, no Autoship nudge | None |
| Treatment | Autoship nudge + promo billboard | 10% off next eligible Fix |

A single pairwise comparison is planned: **Treatment vs. Control**. Because only one comparison is planned against the family-wise error budget, no multiple-comparison correction is needed — sizing uses the initial `alpha = 0.05` directly.

## One-sided test
For Autoship Adoption Rate, a **flat** result (no lift from the nudge+promo experience) and a **negative** result (the nudge+promo experience underperforms BAU) lead to the identical rollout decision: do not roll out, continue with BAU. There is no decision on the table that requires distinguishing "no effect" from "a harmful effect" — so this sizing is **one-sided**, powered only to detect a positive lift over BAU.

## Why the eligible population needs a second gate, beyond the base eligibility criteria
The base eligibility criteria for this test are: Manual clients (not already enrolled in Autoship), who completed First Fix checkout with a Buy 1+ keep rate. Those criteria describe who is broadly *eligible* for the test. But randomization does not happen at First Fix checkout — it happens later, at the moment a client selects a Quick Fix date and clicks "Schedule a Quick Fix," which is when the Autoship nudge (and therefore Control/Treatment assignment) is actually shown. A client who meets the base eligibility criteria but never reaches that click point is never actually randomized, and including them in the sizing population overstates both the true daily flow of randomized clients and dilutes the baseline conversion rate with clients who never had a chance to convert in the first place.

**Can this click be tracked directly?** Checked against `curated.product_tracking_events`: a click-level event exists (`schema='select', screen_view_name='anchored_fix_scheduling', action_name='schedule_fix'`), but its coverage is far too sparse to use directly — only around 8% of the base-eligible population shows this event at all, while a much larger share (roughly 46%) can be independently confirmed to have gone on to complete an actual subsequent Fix checkout, which is only possible if they first reached scheduling. Since the checkout-completion fact alone already exceeds the click event's coverage by a wide margin, the click event under-instruments true reach and is not reliable as a standalone gate.

**The proxy used instead:** a client is treated as having reached the nudge if *either* (a) they show a fresh Autoship demand event within the maturation window (adopting Autoship necessarily requires having reached the nudge, so every adopter trivially qualifies), *or* (b) they complete any subsequent Fix checkout within the maturation window, regardless of fulfillment method (completing a checkout necessarily requires having first reached scheduling). Both conditions are fully-resolved facts sourced from transaction and subscription-history tables, not client-side event tracking, so this proxy is immune to the coverage gaps that make the raw click event unreliable. It still likely **undercounts** true reach to some degree: a client who reached the nudge, declined Autoship, and simply hasn't yet completed their next (manually-scheduled) Fix checkout within the window would be missed by this proxy — so the population and rates below should be read as a conservative refinement, not an exact measurement of the true randomized population.

## Metric definition
**Autoship Adoption Rate** = share of clients confirmed to have reached the Autoship nudge who show a fresh Autoship demand event within a 90-day window following their First Fix checkout.

A client's Autoship history is tracked in `curated.client_pulse_journal`, a daily journal (one row per day any tracked client attribute changes) carrying `last_autoship_demand_ts` — the timestamp of that client's most recent Autoship demand event as of that journal row. A client is counted as **adopted** if, scanning their full journal history, the *earliest* `last_autoship_demand_ts` value that is itself later than their First Fix checkout date falls within 90 days of that checkout. Requiring the timestamp to be strictly *after* First Fix checkout (not merely populated) excludes clients carrying a pre-existing Autoship demand timestamp unrelated to this test's nudge.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
COHORT_START = '2025-08-01'
MATURATION_DAYS = 90  # days to wait for a client's Autoship demand event or subsequent checkout to resolve

# Design parameters (2-cell test, single comparison, no multiple-comparison correction)
ALPHA = 0.05  # single comparison: Treatment vs. Control, no Bonferroni adjustment needed
POWER = 0.80
TWO_SIDED = False  # one-sided: only a positive lift over BAU changes the rollout decision
N_ARMS = 2
SPLIT = 0.5  # Control and Treatment are equal-sized arms
MDE_GRID = [0.02, 0.03, 0.04, 0.05]  # relative lift on Autoship Adoption Rate, Treatment vs. Control

## Step 1 — Confirming the click event's coverage gap

Before building the reach-gated population, this step confirms the click-event coverage claim made above directly against the data: how many base-eligible clients show the `schedule_fix` click event, versus how many can be independently confirmed to have completed a subsequent Fix checkout (which requires having reached scheduling as a precondition).

In [2]:
click_coverage_query = f"""--sql
WITH first_fix AS (
    SELECT client_id, shipment_id, MIN(checkout_date) AS checkout_date_1,
           ARBITRARY(autoship_or_manual) AS autoship_or_manual, SUM(sold_paid_fix_flag) AS n_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1 AND created_date >= DATE '2026-01-01' AND created_date < DATE '2026-03-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date_1 FROM first_fix WHERE autoship_or_manual = 'manual' AND n_kept >= 1
),
click_event AS (
    SELECT DISTINCT client_id
    FROM curated.product_tracking_events
    WHERE screen_view_name = 'anchored_fix_scheduling' AND action_name = 'schedule_fix'
      AND date_in_utc >= DATE '2026-01-01'
),
next_fix AS (
    SELECT client_id, MIN(checkout_date) AS next_fix_checkout_date
    FROM curated.merch_sales_and_feedback
    WHERE fix_number >= 2
    GROUP BY client_id
)
SELECT
    COUNT(*) AS n_eligible,
    SUM(CASE WHEN c.client_id IS NOT NULL THEN 1 ELSE 0 END) AS n_with_click_event,
    SUM(CASE WHEN n.next_fix_checkout_date IS NOT NULL
              AND n.next_fix_checkout_date <= e.checkout_date_1 + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END) AS n_with_completed_next_checkout
FROM eligible e
LEFT JOIN click_event c ON c.client_id = e.client_id
LEFT JOIN next_fix n ON n.client_id = e.client_id
"""

query(click_coverage_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible,n_with_click_event,n_with_completed_next_checkout
0,18750,1422,8605


**Reading this:** the click event's coverage is a small fraction of the population that can be independently confirmed, via completed transactions alone, to have reached scheduling. This confirms the click event under-instruments true reach and motivates using the checkout/demand-based proxy defined above instead.

## Step 2 — Reach rate & Autoship Adoption Rate among clients who reached the nudge

The eligible population is identified from `curated.merch_sales_and_feedback`: each client's earliest `fix_number = 1` shipment, gated to Manual + Buy 1+, with a 90-day maturation cutoff. Within that population, `reached_nudge_gate` flags clients who either adopted (fresh Autoship demand within the window) or completed any subsequent Fix checkout within the window. The Autoship Adoption Rate is then computed only among clients who satisfy that gate.

In [3]:
baseline_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
      AND checkout_date <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
fresh_demand AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.last_autoship_demand_ts > e.checkout_date
    GROUP BY e.client_id
),
next_fix AS (
    SELECT client_id, MIN(checkout_date) AS next_fix_checkout_date
    FROM curated.merch_sales_and_feedback
    WHERE fix_number >= 2
    GROUP BY client_id
),
joined AS (
    SELECT
        e.client_id,
        DATE_TRUNC('month', e.checkout_date) AS month,
        e.checkout_date,
        CASE WHEN f.first_fresh_demand_ts IS NOT NULL
              AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS adopted_autoship,
        CASE WHEN (f.first_fresh_demand_ts IS NOT NULL
                   AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY)
                  OR (n.next_fix_checkout_date IS NOT NULL
                      AND n.next_fix_checkout_date <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY)
             THEN 1 ELSE 0 END AS reached_nudge_gate
    FROM eligible e
    LEFT JOIN fresh_demand f ON f.client_id = e.client_id
    LEFT JOIN next_fix n ON n.client_id = e.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    SUM(reached_nudge_gate) AS n_reached_nudge_gate,
    CAST(SUM(reached_nudge_gate) AS DOUBLE) / COUNT(*) AS reach_rate,
    SUM(adopted_autoship) AS n_adopted,
    CAST(SUM(adopted_autoship) AS DOUBLE) / NULLIF(SUM(reached_nudge_gate), 0) AS autoship_adoption_rate_among_reached
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

baseline_df = query(baseline_query)
baseline_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,n_reached_nudge_gate,reach_rate,n_adopted,autoship_adoption_rate_among_reached
0,2026-05-01,27,8284,4243,0.512192,2069,0.487627
1,2026-04-01,30,10258,5234,0.510236,2427,0.463699
2,2026-03-01,31,10857,5352,0.492954,2350,0.439088
3,2026-02-01,28,8799,4561,0.518354,2105,0.461522
4,2026-01-01,31,10367,5256,0.506993,2699,0.513508
5,2025-12-01,31,8916,4791,0.537349,2983,0.622626
6,2025-11-01,30,7250,3424,0.472276,1462,0.426986
7,2025-10-01,31,9202,4084,0.443817,1581,0.387120
8,2025-09-01,30,9273,4028,0.434379,1471,0.365194
9,2025-08-01,31,7167,3131,0.436863,1031,0.329288


In [4]:
# Reference month = most recent calendar month fully past the 90-day maturation cutoff as of this run.
REFERENCE_MONTH = '2026-04-01'
ref = baseline_df[baseline_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

REACH_RATE = float(ref['reach_rate'][0])
BASELINE_RATE = float(ref['autoship_adoption_rate_among_reached'][0])

print(f"REACH_RATE = {REACH_RATE:.4f}  |  BASELINE_RATE (among reached) = {BASELINE_RATE:.4f}")
ref.T

REACH_RATE = 0.5102  |  BASELINE_RATE (among reached) = 0.4637


,0
month,2026-04-01
days_observed,30
n_eligible,10258
n_reached_nudge_gate,5234
reach_rate,0.510236
n_adopted,2427
autoship_adoption_rate_among_reached,0.463699


**Reading this:** roughly half of the base-eligible population is confirmed to have reached the nudge within the 90-day window, and among that reached population, the Autoship Adoption Rate is meaningfully higher than the rate measured across the full base-eligible population — consistent with the base-eligible rate being diluted by clients who never had the opportunity to convert.

## Step 3 — A fresher, reach-adjusted daily volume read

Daily eligible volume doesn't need the 90-day maturation wait the adoption-rate read needs, so it's measured off the most recent complete month, then scaled by the matured cohort's `REACH_RATE` to approximate the daily rate at which clients actually reach the nudge (as opposed to the raw rate at which they become base-eligible).

In [5]:
volume_query = """--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-05-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
),
month_days AS (
    SELECT DATE_TRUNC('month', checkout_date) AS month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM eligible
    GROUP BY 1
)
SELECT
    DATE_TRUNC('month', e.checkout_date) AS month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM eligible e
JOIN month_days md ON DATE_TRUNC('month', e.checkout_date) = md.month
GROUP BY DATE_TRUNC('month', e.checkout_date), md.days_observed
ORDER BY 1 DESC
"""

volume_df = query(volume_query)
volume_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,eligible_per_day
0,2026-08-01,25,19038,761.5
1,2026-07-01,31,17779,573.5
2,2026-06-01,30,12146,404.9
3,2026-05-01,31,6820,220.0
4,2026-04-01,1,10,10.0


In [6]:
# Volume reference month = most recent *complete* calendar month (no maturation gate needed for volume).
VOLUME_REFERENCE_MONTH = '2026-07-01'
vol_ref = volume_df[volume_df['month'].astype(str).str.startswith(VOLUME_REFERENCE_MONTH)].reset_index(drop=True)

DAILY_ELIGIBLE_RAW = float(vol_ref['eligible_per_day'][0])
DAILY_ELIGIBLE = DAILY_ELIGIBLE_RAW * REACH_RATE  # scale by the matured cohort's reach rate

print(f"DAILY_ELIGIBLE_RAW ({VOLUME_REFERENCE_MONTH[:7]}) = {DAILY_ELIGIBLE_RAW:,.1f}  |  DAILY_ELIGIBLE, reach-adjusted = {DAILY_ELIGIBLE:,.1f}")

DAILY_ELIGIBLE_RAW (2026-07) = 573.5  |  DAILY_ELIGIBLE, reach-adjusted = 292.6


**Reading this:** the reach-adjusted daily volume is roughly half the raw base-eligible volume, consistent with `REACH_RATE`. This is an approximation — it assumes the reach rate observed in the matured reference month carries forward to the current run rate — but it is more representative of the daily flow of clients who will actually be randomized than the raw base-eligible volume would be on its own.

## Step 4 — Sample size & duration

`n_total_statsmodels` sizes a single pairwise 50/50 comparison (Treatment vs. Control); `n_treatment` is read as the **per-arm** requirement. Each of the 2 arms accrues `DAILY_ELIGIBLE / 2` reached clients per day under the 50/50 split, so `days_required = n_per_arm / (DAILY_ELIGIBLE / 2)`.

In [7]:
def size_table(rel_grid, baseline, daily):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_2arm'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total_2arm', 'days_required', 'weeks_required']]

sided = 'one-sided' if not TWO_SIDED else 'two-sided'
print(f"--- Autoship Adoption Rate among reached clients, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, {sided}, 50/50 split) ---")
size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE)

--- Autoship Adoption Rate among reached clients, Treatment vs. Control (baseline=46.4%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,+2%,0.472973,35798,71596,245,35.0
1,+3%,0.47761,15919,31838,109,15.6
2,+4%,0.482247,8959,17918,62,8.9
3,+5%,0.486884,5736,11472,40,5.7


**Reading this:** despite the reach-adjusted daily volume being roughly half of the raw base-eligible volume, required duration at a given MDE is comparable to or shorter than sizing on the raw base-eligible population would produce — because removing clients who never had a chance to convert nearly doubles the baseline rate, and for a fixed *relative* MDE, a higher baseline needs a smaller sample to detect the same relative lift. The two effects (lower volume, higher baseline) work in opposite directions, and in this case the baseline effect dominates. No harm/guardrail grid is computed here: sizing for a one-sided positive MDE does not symmetrically size for detecting harm, and downside risk on this metric is out of scope for this sizing exercise.

## Step 5 — Summary for the Experiment Design doc

Headline MDE below is a **placeholder 4% relative lift** on Autoship Adoption Rate (Treatment vs. Control) — the second-largest value in the Step 4 grid.

In [8]:
TARGET_REL_MDE = 0.04  # placeholder: 4% relative lift on Autoship Adoption Rate, Treatment vs. Control (second-largest value in the Step 4 grid)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'Autoship Adoption Rate among clients who reached the nudge (Treatment vs. Control)',
    'Population': 'Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, AND confirmed to have reached the Autoship nudge (adopted, or completed a subsequent Fix checkout, within 90 days)',
    'Baseline Value': f"{BASELINE_RATE:.1%} ({REFERENCE_MONTH[:7]}, {MATURATION_DAYS}-day matured cohort, among reached clients)",
    'Reach Rate': f"{REACH_RATE:.1%} of base-eligible clients confirmed to reach the nudge within {MATURATION_DAYS} days",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE:,.0f} / day ({VOLUME_REFERENCE_MONTH[:7]} raw volume, scaled by reach rate)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total (2 arms)': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Autoship Adoption Rate among clients who reached the nudge (Treatment vs. Control)
Population,"Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, AND confirmed to have reached the Autoship nudge (adopted, or completed a subsequent Fix checkout, within 90 days)"
Baseline Value,"46.4% (2026-04, 90-day matured cohort, among reached clients)"
Reach Rate,51.0% of base-eligible clients confirmed to reach the nudge within 90 days
Daily Eligible Volume,"293 / day (2026-07 raw volume, scaled by reach rate)"
Minimum Detectable Effect,+4% relative (0.464 -> 0.482)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)


## Bottom line

- **The base eligibility criteria (Manual, First Fix complete, Buy 1+) describe who qualifies for the test, not who is actually randomized.** Randomization happens later, at the "Schedule a Quick Fix" click — and roughly half of the base-eligible population cannot be confirmed to reach that point within 90 days.
- **Direct click-level tracking is not reliable for this purpose:** its coverage is far below what can be independently confirmed through completed transactions alone, so this notebook uses a deterministic proxy (adopted, or completed any subsequent Fix checkout) instead. That proxy still likely undercounts true reach somewhat, since a decliner who hasn't yet completed a slow-to-resolve manual reorder is missed.
- **Measuring the Autoship Adoption Rate only among confirmed-reached clients nearly doubles the baseline rate** compared to measuring it across the full base-eligible population, since the broader population's rate is diluted by clients who never had the opportunity to convert.
- **Net effect on sizing:** the reach-adjusted daily volume is roughly half the raw base-eligible volume, but the higher baseline more than compensates — required duration at the same MDE is comparable to or shorter than sizing on the raw, ungated population.
- At a **4% relative lift** (the placeholder MDE above), the experiment needs the sample size and duration shown in Step 5.